
# VoxelMorph – Distanz **Pred-Lesion ↔ GT-NeedleTip** (im Needle-Bild / fixed-space)

Dieses Notebook ist für deinen **VXM-Workflow** gedacht:

- **moving** = Initialaufnahme
- **fixed** = Needle-Aufnahme
- **Flow** kommt aus der VoxelMorph-Inference und liegt auf dem **fixed/Needle-Gitter**
- die **Lesion** wird aus dem **Initialraum** in den **Needle-Raum** propagiert
- der **GT-NeedleTip** wird **direkt** aus der **Needle-Annotation** geladen
- anschließend wird die Distanz zwischen **predicted lesion** und **GT needle tip** in **mm** berechnet

## Wichtige Annahme zur Flow-Konvention

Dieses Notebook nimmt standardmäßig an, dass der gespeicherte Flow ein **backward map auf dem fixed-Gitter** ist:

\[
x_{moving} = x_{fixed} + flow(x_{fixed})
\]

Gesucht ist der Punkt im fixed/needle-Raum zu einem gegebenen Punkt im moving/initial-Raum.  
Dafür wird numerisch invertiert:

\[
x_f^{k+1} = x_m - flow(x_f^k)
\]

Das passt zur üblichen VoxelMorph-Interpretation für `moving -> fixed`, wenn `warp = model(moving, fixed)` erzeugt wird.

## Ergebnis

Pro Fall bekommst du:

- `xyz_lesion_pred_mm` = vorhergesagte Läsionsposition im Needle-Raum
- `xyz_tip_gt_mm` = GT-NeedleTip im Needle-Raum
- `dist_pred_lesion_gt_needletip_mm`


**Hinweis (angepasste Visualisierung):** Der Single-Case-Plot zeigt jetzt standardmäßig den **vollen Needle-Slice** statt eines kleinen Crops. Dadurch bleibt die Aufnahme deutlich schärfer und besser interpretierbar. Die Box-Größe ist nur noch ein Overlay um die Punkte, kein Zoom mehr.


In [1]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
import ipywidgets as widgets

plt.rcParams.update({"figure.figsize": (6, 6), "axes.grid": False})
np.set_printoptions(suppress=True, precision=3)


In [2]:

# ============================================================
# Basispfade
# ============================================================

VXM_ROOT = Path("/home/students/studunals1/biopsy_project/vxm").resolve()
CEPH_ROOT = Path("/mnt/ceph/vol_02_home_students/studunals1/biopsy_project").resolve()

NOTEBOOK_DIR = (VXM_ROOT / "distances").resolve()
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_TEST_CSV = (VXM_ROOT / "Analysis" / "val_lesion_0.csv").resolve()

DEFAULT_INF_BASE_DIR_CANDS = [
    (VXM_ROOT / "inference").resolve(),
    (VXM_ROOT / "ncc" / "lesion" / "inference").resolve(),
    (VXM_ROOT / "results" / "inference").resolve(),
]

DEFAULT_INF_BASE_DIR = None
for c in DEFAULT_INF_BASE_DIR_CANDS:
    if c.exists():
        DEFAULT_INF_BASE_DIR = c
        break
if DEFAULT_INF_BASE_DIR is None:
    DEFAULT_INF_BASE_DIR = DEFAULT_INF_BASE_DIR_CANDS[0]

print("VXM_ROOT         :", VXM_ROOT)
print("CEPH_ROOT        :", CEPH_ROOT)
print("NOTEBOOK_DIR     :", NOTEBOOK_DIR)
print("DEFAULT_TEST_CSV :", DEFAULT_TEST_CSV)
print("DEFAULT_INF_BASE :", DEFAULT_INF_BASE_DIR)


VXM_ROOT         : /mnt/ceph/vol_02_home_students/studunals1/biopsy_project/vxm
CEPH_ROOT        : /mnt/ceph/vol_02_home_students/studunals1/biopsy_project
NOTEBOOK_DIR     : /mnt/ceph/vol_02_home_students/studunals1/biopsy_project/vxm/distances
DEFAULT_TEST_CSV : /mnt/ceph/vol_02_home_students/studunals1/biopsy_project/vxm/Analysis/val_lesion_0.csv
DEFAULT_INF_BASE : /mnt/ceph/vol_02_home_students/studunals1/biopsy_project/vxm/inference


In [3]:

# ============================================================
# Helper: Pfade + CSV
# ============================================================

def resolve_existing_path(p_str: str, extra_roots=None) -> Path:
    if p_str is None or (isinstance(p_str, float) and np.isnan(p_str)):
        raise FileNotFoundError(f"Pfad ist leer/NaN: {p_str}")

    p = Path(str(p_str)).expanduser()

    if p.is_absolute() and p.exists():
        return p.resolve()

    roots = [VXM_ROOT]
    if extra_roots:
        roots += [Path(r) for r in extra_roots]

    for r in roots:
        cand = (r / p).resolve()
        if cand.exists():
            return cand

    raise FileNotFoundError(f"Pfad nicht gefunden: {p_str} (probiert roots={roots})")


def load_pairs_csv(csv_path: Path) -> pd.DataFrame:
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV nicht gefunden: {csv_path}")

    df_ = pd.read_csv(csv_path)
    required = ["moving", "fixed", "patient_id", "study_id", "breast_side"]
    missing = [c for c in required if c not in df_.columns]
    if missing:
        raise ValueError(
            f"CSV hat nicht alle benötigten Spalten.\n"
            f"Fehlend: {missing}\n"
            f"Vorhanden: {list(df_.columns)}\n"
            f"CSV: {csv_path}"
        )
    return df_


In [4]:

# ============================================================
# Annotationen laden / finden
# ============================================================

CANDIDATE_LABELS_LESION = ["Lesion", "Läsion", "Tumor"]
CANDIDATE_LABELS_NEEDLE = ["NeedleTip", "Needletip", "Needle", "Nadelspitze", "Nadel", "Tip", "Spitze"]

def load_points_from_json(json_path: Path):
    json_path = Path(json_path)
    with json_path.open("r") as f:
        data = json.load(f)

    annotations = data.get("annotations", [])
    points = []

    for ann in annotations:
        for p in ann.get("points", []):
            lbl = p.get("name") or p.get("label")
            coords = p.get("coords_transformed") or p.get("coords")
            if lbl is None or coords is None or len(coords) < 3:
                continue
            points.append({"label": lbl, "xyz_mm": np.array(coords[:3], dtype=float)})

    return points


def _json_has_label(json_path: Path, candidate_labels):
    try:
        pts = load_points_from_json(json_path)
    except Exception:
        return False

    cand = {c.lower() for c in candidate_labels}
    for p in pts:
        lbl = (p.get("label") or "").lower()
        if lbl in cand:
            return True
    return False


def get_point_for_labels(json_path: Path, candidate_labels):
    pts = load_points_from_json(json_path)
    labels_seen = [p["label"] for p in pts]
    cand = {c.lower() for c in candidate_labels}

    for p in pts:
        if (p["label"] or "").lower() in cand:
            return p

    raise ValueError(
        f"Kein Punkt mit Label aus {candidate_labels} in {json_path} gefunden.\n"
        f"Verfügbare Labels: {labels_seen}"
    )


def find_annotations_dir_from_volume(vol_path: Path, max_up=8) -> Path:
    p = Path(vol_path).resolve()
    cur = p.parent
    for _ in range(max_up):
        cand = cur / "annotations"
        if cand.is_dir():
            return cand
        cur = cur.parent
    raise FileNotFoundError(f"Kein annotations/-Ordner in Elternpfaden von {vol_path}")


def find_lesion_json_for_initial(anno_dir: Path, candidate_labels=CANDIDATE_LABELS_LESION):
    '''
    Bevorzugt die Lesion-Annotation aus Initial/KMDyn.
    Penalisiert Needle / Marker / Biopsy.
    '''
    anno_dir = Path(anno_dir)
    anno_files = sorted([p for p in anno_dir.iterdir() if p.is_file() and p.name.lower().endswith(".json")])
    if not anno_files:
        raise FileNotFoundError(f"Keine .json-Annotationen in {anno_dir}")

    candidates = []
    for f in anno_files:
        if not _json_has_label(f, candidate_labels):
            continue
        name_l = f.name.lower()
        score = 0
        if "kmdyn" in name_l:
            score += 5
        if "init" in name_l or "initial" in name_l:
            score += 4
        if "needle" in name_l or "nadel" in name_l:
            score -= 4
        if "marker" in name_l:
            score -= 2
        if "biops" in name_l:
            score -= 2
        candidates.append((score, f))

    if not candidates:
        raise RuntimeError(f"Keine passende Initial-Lesion-Annotation in {anno_dir} gefunden.")

    candidates.sort(key=lambda t: t[0], reverse=True)
    best_score, best_file = candidates[0]
    print(f"[INFO] Initial-Lesion-Annotation: {best_file.name} (Score={best_score})")
    return best_file


def find_needletip_json_for_needle(anno_dir: Path, candidate_labels=CANDIDATE_LABELS_NEEDLE):
    '''
    Bevorzugt die Needle/Nadel-Annotation im Needle-Kontext.
    Penalisiert Biopsy / Init / Marker.
    '''
    anno_dir = Path(anno_dir)
    anno_files = sorted([p for p in anno_dir.iterdir() if p.is_file() and p.name.lower().endswith(".json")])
    if not anno_files:
        raise FileNotFoundError(f"Keine .json-Annotationen in {anno_dir}")

    candidates = []
    for f in anno_files:
        if not _json_has_label(f, candidate_labels):
            continue
        name_l = f.name.lower()
        score = 0
        if "needle" in name_l or "nadel" in name_l:
            score += 5
        if "biops" in name_l:
            score -= 3
        if "kmdyn" in name_l:
            score -= 3
        if "init" in name_l or "initial" in name_l:
            score -= 3
        if "marker" in name_l:
            score -= 2
        candidates.append((score, f))

    if not candidates:
        raise RuntimeError(f"Keine passende NeedleTip-Annotation im Needle-Kontext in {anno_dir} gefunden.")

    candidates.sort(key=lambda t: t[0], reverse=True)
    best_score, best_file = candidates[0]
    print(f"[INFO] NeedleTip-Annotation (Needle): {best_file.name} (Score={best_score})")
    return best_file


In [5]:

# ============================================================
# Inference-Run / Session Discovery (VXM-kompatibel)
# ============================================================

def list_session_dirs(run_dir: Path):
    run_dir = Path(run_dir)
    if not run_dir.is_dir():
        return []
    return sorted([d for d in run_dir.iterdir() if d.is_dir() and d.name.startswith("inf_")])


def discover_runs(base_dir: Path):
    '''
    Unterstützt zwei Layouts:
    1) base_dir = .../inference/lesion_ncc_xyz     und darin inf_*
    2) base_dir = .../inference                     und darin mehrere run-Ordner
    '''
    base_dir = Path(base_dir)
    if not base_dir.is_dir():
        return []

    children = sorted([d for d in base_dir.iterdir() if d.is_dir()])
    if any(d.name.startswith("inf_") for d in children):
        return [base_dir]

    return children


def pick_latest_session_dir(run_dir: Path) -> Path:
    run_dir = Path(run_dir)
    if run_dir.name.startswith("inf_"):
        return run_dir

    sess = list_session_dirs(run_dir)
    if sess:
        return sess[-1]

    # Falls run_dir direkt Case-Ordner enthält, nimm run_dir
    return run_dir


def load_flow_for_case(pid, study, side, run_dir: Path, session_dir: Path = None):
    '''
    Erwartet:
      <session_dir>/<pid>_<study>_<side>/flow.nii.gz

    Akzeptiert Flow-Layouts:
      - (X, Y, Z, 3) -> wird zu (3, X, Y, Z)
      - (3, X, Y, Z) -> bleibt so
    '''
    run_dir = Path(run_dir)
    if session_dir is None:
        session_dir = pick_latest_session_dir(run_dir)
    else:
        session_dir = Path(session_dir)

    case_id = f"{pid}_{study}_{side}"
    case_dir = session_dir / case_id
    flow_path = case_dir / "flow.nii.gz"

    if not flow_path.exists():
        raise FileNotFoundError(f"flow.nii.gz nicht gefunden: {flow_path}")

    flow_img = nib.load(str(flow_path))
    flow_np = flow_img.get_fdata().astype(np.float32)

    if flow_np.ndim != 4:
        raise ValueError(f"Flow muss 4D sein, bekam {flow_np.shape} aus {flow_path}")

    if flow_np.shape[-1] == 3:
        flow_np = np.moveaxis(flow_np, -1, 0)   # (3, X, Y, Z)
    elif flow_np.shape[0] == 3:
        pass
    else:
        raise ValueError(f"Unerwartete Flow-Shape {flow_np.shape} in {flow_path}")

    return flow_np, flow_img, case_dir


In [6]:

# ============================================================
# Punkt-Mapping: moving -> fixed via Inversion
# ============================================================

def _sample_flow_trilinear(flow_np: np.ndarray, x: float, y: float, z: float) -> np.ndarray:
    '''
    Trilineares Sampling eines Flows.
    flow_np: (3, X, Y, Z) auf FIXED-Gitter
    '''
    _, X, Y, Z = flow_np.shape

    x = float(np.clip(x, 0, X - 1))
    y = float(np.clip(y, 0, Y - 1))
    z = float(np.clip(z, 0, Z - 1))

    x0 = int(np.floor(x)); x1 = min(x0 + 1, X - 1)
    y0 = int(np.floor(y)); y1 = min(y0 + 1, Y - 1)
    z0 = int(np.floor(z)); z1 = min(z0 + 1, Z - 1)

    xd = x - x0
    yd = y - y0
    zd = z - z0

    def f(ix, iy, iz):
        return flow_np[:, ix, iy, iz]

    c000 = f(x0, y0, z0); c100 = f(x1, y0, z0)
    c010 = f(x0, y1, z0); c110 = f(x1, y1, z0)
    c001 = f(x0, y0, z1); c101 = f(x1, y0, z1)
    c011 = f(x0, y1, z1); c111 = f(x1, y1, z1)

    c00 = c000 * (1 - xd) + c100 * xd
    c10 = c010 * (1 - xd) + c110 * xd
    c01 = c001 * (1 - xd) + c101 * xd
    c11 = c011 * (1 - xd) + c111 * xd

    c0 = c00 * (1 - yd) + c10 * yd
    c1 = c01 * (1 - yd) + c11 * yd

    return (c0 * (1 - zd) + c1 * zd).astype(np.float32)


def moving_point_mm_to_fixed_mm_via_inversion(
    point_xyz_mm_moving: np.ndarray,
    moving_pre_path: Path,
    fixed_pre_path: Path,
    flow_np_fixedgrid: np.ndarray,
    n_iter: int = 20,
):
    '''
    Backward map auf fixed grid:
        x_m = x_f + flow(x_f)

    Gesucht: x_f gegeben x_m
    Fixpunktiteration:
        x_f^{k+1} = x_m - flow(x_f^k)
    '''
    mov_img = nib.load(str(moving_pre_path))
    fix_img = nib.load(str(fixed_pre_path))

    aff_mov_inv = np.linalg.inv(mov_img.affine)
    x_m = np.array(
        nib.affines.apply_affine(aff_mov_inv, np.array(point_xyz_mm_moving, dtype=float)),
        dtype=float,
    )

    x_f = x_m.copy()
    Xf, Yf, Zf = fix_img.shape

    for _ in range(int(n_iter)):
        xs = float(np.clip(x_f[0], 0, Xf - 1))
        ys = float(np.clip(x_f[1], 0, Yf - 1))
        zs = float(np.clip(x_f[2], 0, Zf - 1))
        disp = _sample_flow_trilinear(flow_np_fixedgrid, xs, ys, zs)
        x_f = x_m - disp

    pred_xyz_mm_fixed = np.array(nib.affines.apply_affine(fix_img.affine, x_f), dtype=float)

    debug = {
        "x_moving_vox": x_m,
        "x_fixed_vox_final": x_f,
    }
    return pred_xyz_mm_fixed, debug


In [7]:

# ============================================================
# Visualisierung (volle Needle-Slice, kein Crop-Zoom)
# ============================================================

def mm_to_vox(img: nib.Nifti1Image, xyz_mm):
    aff_inv = np.linalg.inv(img.affine)
    return np.array(nib.affines.apply_affine(aff_inv, np.array(xyz_mm, dtype=float)), dtype=float)

def show_points_on_fixed_volume(
    fixed_path,
    xyz_pred_mm,
    xyz_gt_mm,
    axis="z",
    patch=41,
    title="",
    dist_mm=None,
):
    fixed_path = Path(fixed_path)
    img = nib.load(str(fixed_path))
    vol = img.get_fdata().astype(np.float32)

    p_pred = mm_to_vox(img, xyz_pred_mm)
    p_gt = mm_to_vox(img, xyz_gt_mm)

    if axis == "z":
        sl = int(np.clip(round((p_pred[2] + p_gt[2]) / 2), 0, vol.shape[2] - 1))
        img2d = vol[:, :, sl]
        x_pred, y_pred = p_pred[0], p_pred[1]
        x_gt, y_gt = p_gt[0], p_gt[1]
    elif axis == "y":
        sl = int(np.clip(round((p_pred[1] + p_gt[1]) / 2), 0, vol.shape[1] - 1))
        img2d = vol[:, sl, :]
        x_pred, y_pred = p_pred[0], p_pred[2]
        x_gt, y_gt = p_gt[0], p_gt[2]
    elif axis == "x":
        sl = int(np.clip(round((p_pred[0] + p_gt[0]) / 2), 0, vol.shape[0] - 1))
        img2d = vol[sl, :, :]
        x_pred, y_pred = p_pred[1], p_pred[2]
        x_gt, y_gt = p_gt[1], p_gt[2]
    else:
        raise ValueError("axis muss x, y oder z sein")

    plt.figure(figsize=(8, 8))
    plt.imshow(img2d.T, cmap="gray", origin="lower", interpolation="none")

    plt.scatter(
        [x_pred], [y_pred],
        s=90, marker="x", linewidths=2.0,
        label="Pred Lesion"
    )
    plt.scatter(
        [x_gt], [y_gt],
        s=120, marker="+", linewidths=2.2,
        label="GT NeedleTip"
    )

    if patch and patch > 0:
        half = patch / 2.0
        for x, y, col in [(x_pred, y_pred, "tab:blue"), (x_gt, y_gt, "tab:orange")]:
            x0, y0 = x - half, y - half
            x1, y1 = x + half, y + half
            plt.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0], "--", color=col, lw=1)

    title_line = f"{title}\nAxis={axis} Slice={sl}"
    if dist_mm is not None and np.isfinite(dist_mm):
        title_line += f" | Dist={dist_mm:.2f} mm"
    plt.title(title_line)

    plt.axis("off")
    plt.legend(loc="upper right")
    plt.show()


In [8]:

# ============================================================
# Single Case / Alle Fälle
# ============================================================

def compute_pred_lesion_gt_needletip_for_case(
    df: pd.DataFrame,
    case_idx: int,
    run_dir: Path,
    session_dir: Path = None,
    inv_iters: int = 20,
    verbose: bool = True,
):
    if not (0 <= case_idx < len(df)):
        raise IndexError(f"case_idx {case_idx} außerhalb 0..{len(df)-1}")

    row = df.iloc[case_idx]
    pid   = row["patient_id"]
    study = row["study_id"]
    side  = row["breast_side"]

    moving_initial_pre = resolve_existing_path(row["moving"], extra_roots=[CEPH_ROOT])
    fixed_needle_pre   = resolve_existing_path(row["fixed"],  extra_roots=[CEPH_ROOT])

    if verbose:
        print(f"\n=== Fall {case_idx:02d} ===")
        print(f"patient_id : {pid}")
        print(f"study_id   : {study}")
        print(f"breast_side: {side}")
        print("moving_initial_pre:", moving_initial_pre)
        print("fixed_needle_pre  :", fixed_needle_pre)

    anno_dir = find_annotations_dir_from_volume(fixed_needle_pre)
    if verbose:
        print("Anno-Dir:", anno_dir)

    lesion_json = find_lesion_json_for_initial(anno_dir)
    lesion_info = get_point_for_labels(lesion_json, CANDIDATE_LABELS_LESION)
    xyz_lesion_init_mm = lesion_info["xyz_mm"]

    tip_json = find_needletip_json_for_needle(anno_dir)
    tip_info = get_point_for_labels(tip_json, CANDIDATE_LABELS_NEEDLE)
    xyz_tip_gt_mm = tip_info["xyz_mm"]

    if verbose:
        print("Lesion-GT (Initial):", lesion_info)
        print("NeedleTip-GT (Needle):", tip_info)

    flow_np, _, case_dir = load_flow_for_case(pid, study, side, run_dir=run_dir, session_dir=session_dir)
    xyz_lesion_pred_mm, dbg = moving_point_mm_to_fixed_mm_via_inversion(
        point_xyz_mm_moving=xyz_lesion_init_mm,
        moving_pre_path=moving_initial_pre,
        fixed_pre_path=fixed_needle_pre,
        flow_np_fixedgrid=flow_np,
        n_iter=inv_iters,
    )

    dist_mm = float(np.linalg.norm(xyz_lesion_pred_mm - xyz_tip_gt_mm))

    if verbose:
        print("Pred Lesion (Needle, mm):", xyz_lesion_pred_mm)
        print("GT NeedleTip (Needle, mm):", xyz_tip_gt_mm)
        print(f"Distanz = {dist_mm:.2f} mm")
        print("Case dir:", case_dir)

    return {
        "case_idx": int(case_idx),
        "patient_id": pid,
        "study_id": study,
        "breast_side": side,
        "dist_pred_lesion_gt_needletip_mm": dist_mm,
        "xyz_lesion_init_mm": xyz_lesion_init_mm,
        "xyz_lesion_pred_mm": xyz_lesion_pred_mm,
        "xyz_tip_gt_mm": xyz_tip_gt_mm,
        "moving_initial_pre_path": str(moving_initial_pre),
        "fixed_needle_pre_path": str(fixed_needle_pre),
        "debug": dbg,
    }


def run_all_cases(
    df: pd.DataFrame,
    run_dir: Path,
    session_dir: Path,
    output_csv: Path,
    inv_iters: int = 20,
):
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    rows_out = []
    n = len(df)

    for idx in range(n):
        row = df.iloc[idx]
        pid   = row["patient_id"]
        study = row["study_id"]
        side  = row["breast_side"]

        dist_mm = np.nan
        try:
            res = compute_pred_lesion_gt_needletip_for_case(
                df=df,
                case_idx=idx,
                run_dir=run_dir,
                session_dir=session_dir,
                inv_iters=inv_iters,
                verbose=False,
            )
            dist_mm = res["dist_pred_lesion_gt_needletip_mm"]
            print(f"[{idx+1:03d}/{n}] pid={pid} study={study} side={side}  dist={dist_mm:.2f} mm")
        except Exception as e:
            print(f"[{idx+1:03d}/{n}] pid={pid} study={study} side={side}  FEHLER -> {repr(e)}")

        rows_out.append({
            "patient_id": pid,
            "study_id": study,
            "breast_side": side,
            "dist_pred_lesion_gt_needletip_mm": dist_mm,
        })

    df_out = pd.DataFrame(rows_out)
    df_out.to_csv(output_csv, index=False)
    print("\nFertig. CSV geschrieben nach:", output_csv)
    return df_out


In [ ]:

# ============================================================
# Widgets / UI
# ============================================================

csv_text = widgets.Text(
    value=str(DEFAULT_TEST_CSV),
    description="CSV:",
    layout=widgets.Layout(width="88%"),
)

inf_base_text = widgets.Text(
    value=str(DEFAULT_INF_BASE_DIR),
    description="Inf base:",
    layout=widgets.Layout(width="88%"),
)

reload_button = widgets.Button(
    description="Neu laden",
    tooltip="CSV + Inference-Ordner scannen",
)

run_dropdown = widgets.Dropdown(
    options=[],
    description="Run:",
    layout=widgets.Layout(width="88%"),
)

session_dropdown = widgets.Dropdown(
    options=[],
    description="Session:",
    layout=widgets.Layout(width="88%"),
)

case_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description="Case:",
    continuous_update=False,
)

axis_dropdown = widgets.Dropdown(
    options=[("z (axial)", "z"), ("y (coronal)", "y"), ("x (sagittal)", "x")],
    value="z",
    description="Axis:",
)

patch_slider = widgets.IntSlider(
    value=41,
    min=0,
    max=121,
    step=2,
    description="Box:",
    continuous_update=False,
)

inv_iters_slider = widgets.IntSlider(
    value=20,
    min=1,
    max=60,
    step=1,
    description="Inv iters:",
    continuous_update=False,
)

output_csv_text = widgets.Text(
    value=str(NOTEBOOK_DIR / "dist_pred_lesion_gt_needletip_vxm.csv"),
    description="Out CSV:",
    layout=widgets.Layout(width="88%"),
)

run_single_button = widgets.Button(description="Single Case", button_style="info")
run_all_button = widgets.Button(description="Alle Fälle -> CSV", button_style="success")

output_area = widgets.Output()
_df = None

def refresh_runs_and_sessions():
    global _df

    csv_path = Path(csv_text.value).expanduser()
    _df = load_pairs_csv(csv_path)

    case_slider.max = max(0, len(_df) - 1)
    case_slider.value = 0 if len(_df) > 0 else 0

    base_dir = Path(inf_base_text.value).expanduser()
    runs = discover_runs(base_dir)

    if not runs:
        run_dropdown.options = [("– keine Runs gefunden –", None)]
        session_dropdown.options = [("–", None)]
        return

    run_dropdown.options = [(d.name, d) for d in runs]
    run_dropdown.value = runs[0]

def refresh_sessions_for_run(run_dir: Path):
    if run_dir is None:
        session_dropdown.options = [("–", None)]
        session_dropdown.value = None
        return

    run_dir = Path(run_dir)

    if run_dir.name.startswith("inf_"):
        session_dropdown.options = [("(ausgewähltes inf_*)", run_dir)]
        session_dropdown.value = run_dir
        return

    sessions = list_session_dirs(run_dir)
    if sessions:
        opts = [("LATEST (automatisch)", None)] + [(s.name, s) for s in sessions]
        session_dropdown.options = opts
        session_dropdown.value = None
        return

    session_dropdown.options = [("(Run dir direkt)", run_dir)]
    session_dropdown.value = run_dir

def get_selected_run_and_session():
    run_dir = run_dropdown.value
    session_dir = session_dropdown.value
    return run_dir, session_dir

def on_run_changed(change):
    new_run = change["new"]
    with output_area:
        clear_output()
        try:
            refresh_sessions_for_run(new_run)
            print("Run geändert:", new_run)
        except Exception as e:
            print("FEHLER beim Aktualisieren der Sessions:")
            print(repr(e))

def on_reload_clicked(_):
    with output_area:
        clear_output()
        try:
            refresh_runs_and_sessions()
            if run_dropdown.value is not None:
                refresh_sessions_for_run(run_dropdown.value)
            print("Neu geladen.")
            print("CSV:", csv_text.value)
            print("Inf base:", inf_base_text.value)
            print("Fälle:", len(_df))
        except Exception as e:
            print("FEHLER beim Laden:")
            print(repr(e))

def on_single_clicked(_):
    with output_area:
        clear_output()
        try:
            if _df is None:
                refresh_runs_and_sessions()

            idx = case_slider.value
            run_dir, sess_dir = get_selected_run_and_session()
            print("Run:", run_dir)
            print("Session:", sess_dir)
            print("Inv iters:", inv_iters_slider.value)

            res = compute_pred_lesion_gt_needletip_for_case(
                df=_df,
                case_idx=idx,
                run_dir=run_dir,
                session_dir=sess_dir,
                inv_iters=inv_iters_slider.value,
                verbose=True,
            )

            print("\nResult summary:")
            for k, v in res.items():
                if isinstance(v, np.ndarray):
                    print(f"  {k}: {v}")
                else:
                    print(f"  {k}: {v}")

            show_points_on_fixed_volume(
                fixed_path=res["fixed_needle_pre_path"],
                xyz_pred_mm=res["xyz_lesion_pred_mm"],
                xyz_gt_mm=res["xyz_tip_gt_mm"],
                axis=axis_dropdown.value,
                patch=patch_slider.value,
                title=f'Case {idx}: {res["patient_id"]}_{res["study_id"]}_{res["breast_side"]}',
                dist_mm=res["dist_pred_lesion_gt_needletip_mm"],
            )
        except Exception as e:
            print("FEHLER bei Single Case:")
            print(repr(e))

def on_all_clicked(_):
    with output_area:
        clear_output()
        try:
            if _df is None:
                refresh_runs_and_sessions()
            run_dir, sess_dir = get_selected_run_and_session()
            out_csv = Path(output_csv_text.value).expanduser()

            print("Run:", run_dir)
            print("Session:", sess_dir)
            print("Inv iters:", inv_iters_slider.value)
            print("Out CSV:", out_csv)

            df_out = run_all_cases(
                df=_df,
                run_dir=run_dir,
                session_dir=sess_dir,
                output_csv=out_csv,
                inv_iters=inv_iters_slider.value,
            )
            display(df_out.head())
        except Exception as e:
            print("FEHLER bei Alle-Fälle-Auswertung:")
            print(repr(e))

reload_button.on_click(on_reload_clicked)
run_dropdown.observe(on_run_changed, names="value")
run_single_button.on_click(on_single_clicked)
run_all_button.on_click(on_all_clicked)

with output_area:
    clear_output()
    try:
        refresh_runs_and_sessions()
        if run_dropdown.value is not None:
            refresh_sessions_for_run(run_dropdown.value)
        print("Initial geladen.")
        print("CSV:", csv_text.value)
        print("Inf base:", inf_base_text.value)
        print("Fälle:", len(_df))
    except Exception as e:
        print("FEHLER beim Initial-Laden:")
        print(repr(e))

ui = widgets.VBox([
    csv_text,
    inf_base_text,
    widgets.HBox([reload_button, inv_iters_slider]),
    run_dropdown,
    session_dropdown,
    widgets.HBox([case_slider, axis_dropdown, patch_slider]),
    output_csv_text,
    widgets.HBox([run_single_button, run_all_button]),
    output_area,
])

display(ui)
